# Features analysis code

## Requirements

In [8]:
 !pip3 install pandas openpyxl scipy

## Imports

In [1]:
import pandas as pd
import openpyxl
import scipy

## Dataframe import

In [2]:
features_df = pd.read_excel("../extracted_features/extracted_features.xlsx")
features_df

,participant_id,gender,age,condition,perceived_topic_intimacy,personal_discomfort,revealing_personal_information
0,4,M,23,1,5,1.571,1.25
1,4,M,23,2,4,1.000,1.00
2,5,M,20,1,3,1.143,1.25
3,5,M,20,2,4,2.429,2.00
4,6,F,27,1,4,1.143,1.00
5,6,F,27,2,3,1.286,2.75
6,7,F,21,1,1,1.000,1.75
7,7,F,21,2,3,1.000,1.75
8,9,M,26,1,4,1.714,2.00
9,9,M,26,2,2,1.286,1.50


## Demographics

In [4]:
# For the computation on demographics
participants_df = features_df.drop_duplicates("participant_id")

### Gender

In [5]:
gender_percent = participants_df["gender"].value_counts(normalize=True) * 100

print(gender_percent)

gender
F    52.0
M    48.0
Name: proportion, dtype: float64


### Age

In [6]:
age_mean = participants_df["age"].mean()
age_std = participants_df["age"].std()
age_min = participants_df["age"].min()
age_max = participants_df["age"].max()

print(f"Mean age = {age_mean:.2f}")
print(f"SD age = {age_std:.2f}")
print(f"Range = {age_min}–{age_max}")

Mean age = 24.52
SD age = 4.93
Range = 18–36


## Post questionnaire analysis

In [13]:
# Helper function
def paired_test(df, variable, alternative, alpha=0.05):
    
    df_paired = df.pivot(
        index="participant_id",
        columns="condition",
        values=variable
    )
    
    diff = df_paired[2] - df_paired[1]   # condition 2 minus condition 1
    
    # Normality test
    shapiro_stat, shapiro_p = scipy.stats.shapiro(diff)
    
    print(f"\n=== {variable} ===")
    print(f"Shapiro p = {shapiro_p:.4f}")
    
    if shapiro_p > alpha:
        print("Normality met → Paired t-test")
        stat, p = scipy.stats.ttest_rel(df_paired[2], df_paired[1])
    else:
        print("Normality violated → Wilcoxon test")
        stat, p = scipy.stats.wilcoxon(df_paired[2], df_paired[1], alternative=alternative)

    print(f"Test statistic = {stat:.3f}")
    print(f"p-value = {p:.4f}")
    
    if p < alpha:
        print(f"Conclusion: {variable} is significantly higher in Condition 2 than in Condition 1.")
    else:
        print(f"Conclusion: There is no statistically significant evidence that {variable} is higher in Condition 2 than in Condition 1.")

In [14]:
paired_test(features_df, "perceived_topic_intimacy", "greater")
paired_test(features_df, "personal_discomfort", "greater")
paired_test(features_df, "revealing_personal_information", "greater")


=== perceived_topic_intimacy ===
Shapiro p = 0.0086
Normality violated → Wilcoxon test
Test statistic = 227.500
p-value = 0.0122
Conclusion: perceived_topic_intimacy is significantly higher in Condition 2 than in Condition 1.

=== personal_discomfort ===
Shapiro p = 0.2384
Normality met → Paired t-test
Test statistic = 4.210
p-value = 0.0003
Conclusion: personal_discomfort is significantly higher in Condition 2 than in Condition 1.

=== revealing_personal_information ===
Shapiro p = 0.0095
Normality violated → Wilcoxon test
Test statistic = 220.000
p-value = 0.0001
Conclusion: revealing_personal_information is significantly higher in Condition 2 than in Condition 1.


## ML classifier

### Data standardization

Note: should be done separatly between train and test
Note: within subject -> data not independent -> Note: within subject -> Grouped cross-validation or Leave-one-subject-out (LOSO) (from sklearn.model_selection import GroupKFold)

LOSO: train on 24 participants and test on 1, repeated 25 times

### Clasifier training and testing